# 00 – Vérification de l'environnement ShopStream
Exécutez toutes les cellules. Chaque étape doit afficher **OK**.

In [ ]:
import os
from shopstream_utils import get_spark, read_table, write_table, DATA_DIR
print("Fichiers de données :", os.listdir(DATA_DIR))

## 1. SparkSession (le premier lancement télécharge les connecteurs, ~1 min)

In [ ]:
spark = get_spark("00-verification")
print("OK - Spark", spark.version)

## 2. Lecture des données générées

In [ ]:
products = spark.read.csv(f"{DATA_DIR}/products.csv", header=True, inferSchema=True)
products.show(5)
print("OK -", products.count(), "produits")

## 3. Connexion PostgreSQL (écriture puis lecture d'une table de test)

In [ ]:
write_table(spark.createDataFrame([(1, "ok")], ["id", "status"]), "healthcheck", mode="overwrite")
read_table(spark, "healthcheck").show()
print("OK - PostgreSQL")

## 4. Connexion Kafka
Nécessite que le topic `reviews_stream` existe (Partie 1 du sujet). Lecture *batch* des messages déjà présents.

In [ ]:
from shopstream_utils import KAFKA_BOOTSTRAP, TOPIC
raw = (spark.read.format("kafka")
       .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
       .option("subscribe", TOPIC)
       .option("startingOffsets", "earliest")
       .load())
raw.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "partition", "offset").show(5, truncate=80)
print("OK - Kafka :", raw.count(), "messages dans le topic")